# Etapa 5 — Transformação com Pandas

**Notebook 1: da estrutura aninhada para a tabela `partidas`**

Este notebook é para *explorar*. Bagunça é permitida, código pode ficar pela metade,
célula pode ser rodada fora de ordem. Quando a transformação estiver decidida, ela
migra para um script limpo em `src/transform/`.

Atalhos úteis:
- `Shift + Enter` — roda a célula e vai para a próxima
- `Ctrl + Enter` — roda a célula e fica nela
- `Esc` depois `B` — cria célula abaixo

In [1]:
from pathlib import Path
import json

import pandas as pd

# Faz o Pandas mostrar tabelas largas sem cortar
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 40)

# O notebook roda a partir da pasta notebooks/, então subimos um nível
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQ_PARTIDAS = RAIZ / "data" / "raw" / "partidas.jsonl"

print("arquivo encontrado:", ARQ_PARTIDAS.exists())
print("tamanho:", round(ARQ_PARTIDAS.stat().st_size / 1024**2), "MB")

arquivo encontrado: True
tamanho: 404 MB


---
## 1. O problema: o Pandas não desaninha sozinho

In [2]:
df_bruto = pd.read_json(ARQ_PARTIDAS, lines=True, nrows=100)

print("shape:", df_bruto.shape)
df_bruto.head(3)

shape: (100, 2)


,metadata,info
0,"{'dataVersion': '2', 'matchId': 'BR1...","{'endOfGameResult': 'GameComplete', ..."
1,"{'dataVersion': '2', 'matchId': 'BR1...","{'endOfGameResult': 'GameComplete', ..."
2,"{'dataVersion': '2', 'matchId': 'BR1...","{'endOfGameResult': 'GameComplete', ..."


Duas colunas. Cada célula guarda um dicionário inteiro.

O JSON tem duas chaves no topo (`metadata` e `info`), e o Pandas criou uma coluna
para cada. Ele parou aí — não desce nos níveis aninhados por conta própria.

In [3]:
# Confirmando: o que tem dentro de uma célula?
celula = df_bruto.iloc[0]["info"]
print(type(celula))
print(list(celula.keys()))

<class 'dict'>
['endOfGameResult', 'gameCreation', 'gameDuration', 'gameEndTimestamp', 'gameId', 'gameMode', 'gameName', 'gameStartTimestamp', 'gameType', 'gameVersion', 'mapId', 'participants', 'platformId', 'queueId', 'teams', 'tournamentCode']


---
## 2. A ferramenta certa: `json_normalize`

`pd.json_normalize` recebe uma lista de dicionários e **achata** a estrutura,
criando uma coluna para cada campo aninhado. O nome da coluna vira o caminho
completo, separado por ponto:

```
{"info": {"gameDuration": 1800}}   →   coluna "info.gameDuration"
```

O que ele **não** achata são as listas. `info.participants` é uma lista de 10
jogadores — ela continua como lista dentro da célula. Isso é proposital: 10 itens
numa linha não cabem numa tabela de grão "partida". Vamos tratá-los separadamente,
na tabela `jogadores`.

In [4]:
# Lendo 100 partidas como lista de dicionários
with open(ARQ_PARTIDAS, encoding="utf-8") as f:
    amostra = [json.loads(linha) for _, linha in zip(range(100), f)]

df = pd.json_normalize(amostra)

print("shape:", df.shape)
print()
for coluna in df.columns:
    print(" ", coluna)

shape: (100, 19)

  metadata.dataVersion
  metadata.matchId
  metadata.participants
  info.endOfGameResult
  info.gameCreation
  info.gameDuration
  info.gameEndTimestamp
  info.gameId
  info.gameMode
  info.gameName
  info.gameStartTimestamp
  info.gameType
  info.gameVersion
  info.mapId
  info.participants
  info.platformId
  info.queueId
  info.teams
  info.tournamentCode


Compare: `read_json` deu **2 colunas**, `json_normalize` deu bem mais.

Repare que `metadata.participants`, `info.participants` e `info.teams` continuam
como listas. São os níveis mais fundos, que viram as outras duas tabelas.

---
## 3. Tarefa 5.2 — construir a tabela `partidas`

Grão: **1 linha = 1 partida**. Só campos que descrevem a partida inteira.

Consulte o seu dicionário de dados (`docs/ETAPA-04-dicionario-de-dados.md`) para
decidir quais colunas entram.

In [5]:
# Agora com o arquivo INTEIRO (leva uns segundos)
with open(ARQ_PARTIDAS, encoding="utf-8") as f:
    todas = [json.loads(linha) for linha in f]

df = pd.json_normalize(todas)
print("partidas carregadas:", len(df))

partidas carregadas: 5000


In [6]:
# TODO 1 — Selecione as colunas da tabela `partidas`.
#
# Você precisa de: identificador da partida, duração, patch, e o horário de início.
# Consulte a lista de colunas impressa acima para pegar os nomes exatos.
#
# Sintaxe: df[["coluna_a", "coluna_b"]] devolve um DataFrame só com essas colunas.
# (repare nos colchetes DUPLOS — a lista de nomes vai dentro do colchete de seleção)

colunas = [
    "metadata.matchId",
    "info.gameDuration",
    "info.gameVersion",
    "info.gameStartTimestamp",
]

partidas = df[colunas].copy()   # .copy() evita um aviso chato do Pandas depois
partidas.head()

,metadata.matchId,info.gameDuration,info.gameVersion,info.gameStartTimestamp
0,BR1_3257404313,2679,16.13.791.5903,1782768470733
1,BR1_3257408108,1370,16.13.791.5903,1782768936423
2,BR1_3257408148,2031,16.13.791.5903,1782769015909
3,BR1_3257410457,2255,16.13.791.5903,1782769418247
4,BR1_3257414747,2532,16.13.791.5903,1782770272912


In [7]:
# TODO 2 — Renomeie as colunas para nomes limpos.
#
# "info.gameDuration" é feio e vai virar nome de coluna no PostgreSQL depois.
# Padrão profissional: minúsculas, sem ponto, sem acento, separado por _
#   info.gameDuration  ->  duracao_segundos
#
# Sintaxe: df.rename(columns={"nome_velho": "nome_novo", ...})

partidas = partidas.rename(columns={
    "metadata.matchId": "id_partida",
    "info.gameDuration": "duracao_segundos",
    "info.gameVersion": "patch_jogo",
    "info.gameStartTimestamp": "inicio_jogo_segundos" 
})

partidas.head()

,id_partida,duracao_segundos,patch_jogo,inicio_jogo_segundos
0,BR1_3257404313,2679,16.13.791.5903,1782768470733
1,BR1_3257408108,1370,16.13.791.5903,1782768936423
2,BR1_3257408148,2031,16.13.791.5903,1782769015909
3,BR1_3257410457,2255,16.13.791.5903,1782769418247
4,BR1_3257414747,2532,16.13.791.5903,1782770272912


In [8]:
# TODO 3 — Aplique os filtros de limpeza D1, D2 e D4 do dicionário de dados.
#
# Em Pandas, filtrar é criar uma máscara booleana e usá-la como índice:
#   df[df["duracao_segundos"] >= 300]
#
# A expressão df["coluna"] >= 300 devolve uma Series de True/False (uma por linha),
# e o df[...] mantém só as linhas True.
#
# Para combinar condições use & (e) e | (ou), SEMPRE com parênteses em volta de cada:
#   df[(df["a"] > 1) & (df["b"] < 5)]
#
# Comece pela D1. Imprima quantas linhas sobraram e confira se bate com o esperado.

antes = len(partidas)
partidas = partidas[partidas["duracao_segundos"] >= 300]
print(f"antes: {antes} | depois: {len(partidas)} | removidas: {antes - len(partidas)}")

antes: 5000 | depois: 4778 | removidas: 222


In [9]:
partidas.shape
list(partidas.columns)

['id_partida', 'duracao_segundos', 'patch_jogo', 'inicio_jogo_segundos']

In [10]:
partidas["inicio_partida"] = pd.to_datetime(partidas["inicio_jogo_segundos"], unit="ms")

In [11]:
partidas.head()
partidas.dtypes

id_partida                         str
duracao_segundos                 int64
patch_jogo                         str
inicio_jogo_segundos             int64
inicio_partida          datetime64[ms]
dtype: object

In [12]:
partidas["patch_jogo"].str.split(".")

0       [16, 13, 791, 5903]
1       [16, 13, 791, 5903]
2       [16, 13, 791, 5903]
3       [16, 13, 791, 5903]
4       [16, 13, 791, 5903]
               ...         
4995    [16, 15, 799, 6036]
4996    [16, 15, 799, 6036]
4997    [16, 15, 799, 6036]
4998    [16, 15, 799, 6036]
4999    [16, 15, 799, 6036]
Name: patch_jogo, Length: 4778, dtype: object

In [13]:
partidas["patch_jogo"].str.split(".").str[0]   # "16"
partidas["patch_jogo"].str.split(".").str[1]   # "13"

0       13
1       13
2       13
3       13
4       13
        ..
4995    15
4996    15
4997    15
4998    15
4999    15
Name: patch_jogo, Length: 4778, dtype: object

In [14]:
partidas["patch"] = partidas["patch_jogo"].str.split(".").str[0] + '.' + partidas["patch_jogo"].str.split(".").str[1]

In [15]:
partidas["patch"].value_counts()


patch
16.14    2528
16.13    2215
16.15      35
Name: count, dtype: int64

In [16]:
partidas = partidas.drop(columns=["patch_jogo", "inicio_jogo_segundos"])

In [17]:
antes = len(partidas)
partidas = partidas[partidas["patch"] != "16.15"]
print(f"antes: {antes} | depois: {len(partidas)} | removidas: {antes - len(partidas)}")

antes: 4778 | depois: 4743 | removidas: 35


In [18]:
ids_ruins = []

In [19]:

for partida in todas:
    for player in partida['info']['participants']:
        if player['teamPosition'] == "":
            ids_ruins.append(partida['metadata']['matchId'])

ids_ruins


['BR1_3257879320',
 'BR1_3257924103',
 'BR1_3258930538',
 'BR1_3259125288',
 'BR1_3260016229',
 'BR1_3260275701',
 'BR1_3261106008',
 'BR1_3262038485',
 'BR1_3262240348',
 'BR1_3264113649',
 'BR1_3265533780',
 'BR1_3266420002',
 'BR1_3267000920']

In [20]:
antes = len(partidas)
partidas = partidas[~partidas["id_partida"].isin(ids_ruins)]
print(f"antes: {antes} | depois: {len(partidas)} | removidas: {antes - len(partidas)}")

antes: 4743 | depois: 4741 | removidas: 2


In [21]:
partidas.to_parquet(RAIZ / "data" / "processed" / "partidas.parquet", index=False)

In [22]:
times =[{
    "id_partida": None,
    "team_id": None,
    "venceu": None,
    "primeiro_barao": None,
    "primeira_torre": None,
    "first_blood": None,
    "primeiro_dragao": None,
    "arauto": None,
    "larvas": None,
    "dragoes_abatidos": None,
    "abates": None,
    "torres_destruidas": None
}]

In [29]:
time = []

for partida in todas:
    for equipe in partida['info']['teams']:
        partida_dict = {
            "id_partida": partida['metadata']['matchId'],
            "team_id": equipe["teamId"],
            "venceu": equipe["win"],
            "primeiro_barao": equipe['objectives']['baron']['first'],
            "primeira_torre": equipe['objectives']['tower']['first'],
            "first_blood": equipe['objectives']['champion']['first'],
            "primeiro_dragao": equipe['objectives']['dragon']['first'],
            "arauto": equipe["objectives"]["riftHerald"]["first"],
            "larvas": equipe["objectives"]["horde"]["first"],
            "dragoes_abatidos": equipe["objectives"]["dragon"]["kills"],
            "abates": equipe["objectives"]["champion"]["kills"],
            "torres_destruidas": equipe["objectives"]["tower"]["kills"],
        }
        time.append(partida_dict)

In [30]:
times = pd.DataFrame(time)

In [31]:
times

,id_partida,team_id,venceu,primeiro_barao,primeira_torre,first_blood,primeiro_dragao,arauto,larvas,dragoes_abatidos,abates,torres_destruidas
0,BR1_3257404313,100,True,True,False,False,False,False,True,3,51,9
1,BR1_3257404313,200,False,False,True,True,True,True,False,4,49,9
2,BR1_3257408108,100,False,False,False,False,False,False,True,0,7,2
3,BR1_3257408108,200,True,True,True,True,True,True,False,3,34,10
4,BR1_3257408148,100,True,True,False,True,True,False,False,4,45,9
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,BR1_3267044249,200,True,True,True,True,True,True,True,4,48,11
9996,BR1_3267048703,100,False,False,False,True,False,False,False,0,8,0
9997,BR1_3267048703,200,True,True,True,False,True,True,True,3,32,10
9998,BR1_3267050064,100,False,False,False,True,False,False,True,0,9,0


In [32]:
antes = len(times)
times = times[times["id_partida"].isin(partidas["id_partida"])]
print(f"antes: {antes} | depois: {len(times)} | removidas: {antes - len(times)}")

antes: 10000 | depois: 9482 | removidas: 518


In [33]:
print(len(times))
print(times.groupby("id_partida").size().value_counts())
print(times["venceu"].mean())

9482
2    4741
Name: count, dtype: int64
0.5


In [34]:
times.groupby("primeiro_barao")["venceu"].agg(["count", "sum", "mean"])

,count,sum,mean
primeiro_barao,,,
False,5883,1839,0.312596
True,3599,2902,0.806335


In [35]:
times["alma_do_dragao"] = times["dragoes_abatidos"] >= 4
times.groupby("alma_do_dragao")["venceu"].agg(["count", "sum", "mean"])

,count,sum,mean
alma_do_dragao,,,
False,8252,3659,0.443408
True,1230,1082,0.879675


In [36]:
times.to_parquet(RAIZ / "data" / "processed" / "times.parquet", index=False)
partidas.to_parquet(RAIZ / "data" / "processed" / "partidas.parquet", index=False)

In [60]:
jogadores = []

for partida in todas:
    id_partida = partida["metadata"]["matchId"]
    for jogador in partida["info"]["participants"]:
        jogadores.append({
            "id_partida": id_partida,          # ← chave estrangeira
            "puuid": jogador["puuid"],         # ← identificador do jogador
            "team_id": jogador["teamId"],
            "venceu": jogador["win"],
            "nome_campeao": jogador["championName"],
            "rota": jogador["teamPosition"],
            "kills": jogador["kills"],
            "deaths": jogador["deaths"],
            "assists": jogador["assists"],
            "ouro": jogador["goldEarned"],
            "cs_minion": jogador["totalMinionsKilled"],
            "cs_jungle": jogador["neutralMinionsKilled"],
            "vision_score": jogador["visionScore"],
        })

jogadores = pd.DataFrame(jogadores)

In [62]:
visao_por_time = (
    jogadores
    .groupby(["id_partida", "team_id"])["vision_score"]
    .sum()
    .reset_index()
)

In [63]:
times = times.merge(
    visao_por_time,
    on=["id_partida", "team_id"],
    how="left",
)

In [64]:
print(len(times))                          # continua 9.482 — merge não pode criar linhas
print(times["vision_score"].isna().sum())  # 0 — nenhum time ficou sem visão

9482
0


In [66]:
times.to_parquet(RAIZ / "data" / "processed" / "times.parquet", index=False)
partidas.to_parquet(RAIZ / "data" / "processed" / "partidas.parquet", index=False)
jogadores.to_parquet(RAIZ / "data" / "processed" / "jogadores.parquet", index=False)